# STEP 1: Install libraries + Set Up Environment

In [1]:
!pip install transformers datasets seqeval accelerate -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
!pip install evaluate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.5 MB/s eta 0:00:00


In [3]:
import numpy as np
import pandas as pd

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForTokenClassification

import evaluate

# STEP 2: Load Dataset Manually

In [5]:
!wget https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-train.conllu
!wget https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-dev.conllu
!wget https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-test.conllu

--2026-04-05 14:00:44--  https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-train.conllu
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 15029817 (14M) [text/plain]
Saving to: ‘en_ewt-ud-train.conllu’

en_ewt-ud-train.con 100%[===================>]  14.33M  --.-KB/s    in 0.08s   

2026-04-05 14:00:45 (186 MB/s) - ‘en_ewt-ud-train.conllu’ saved [15029817/15029817]

--2026-04-05 14:00:45--  https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-dev.conllu
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request s

In [6]:
def read_conllu(file_path):
    sentences = []
    labels = []

    with open(file_path, "r", encoding="utf-8") as f:
        tokens = []
        pos_tags = []

        for line in f:
            if line.startswith("#") or line.strip() == "":
                if tokens:
                    sentences.append(tokens)
                    labels.append(pos_tags)
                    tokens = []
                    pos_tags = []
                continue

            parts = line.strip().split("\t")
            if len(parts) > 3:
                tokens.append(parts[1])
                pos_tags.append(parts[3])  # UPOS

    return sentences, labels

train_texts, train_labels = read_conllu("en_ewt-ud-train.conllu")
val_texts, val_labels = read_conllu("en_ewt-ud-dev.conllu")
test_texts, test_labels = read_conllu("en_ewt-ud-test.conllu")

In [7]:
print(train_texts[0])
print(train_labels[0])

['Al', '-', 'Zaman', ':', 'American', 'forces', 'killed', 'Shaikh', 'Abdullah', 'al', '-', 'Ani', ',', 'the', 'preacher', 'at', 'the', 'mosque', 'in', 'the', 'town', 'of', 'Qaim', ',', 'near', 'the', 'Syrian', 'border', '.']
['PROPN', 'PUNCT', 'PROPN', 'PUNCT', 'ADJ', 'NOUN', 'VERB', 'PROPN', 'PROPN', 'PROPN', 'PUNCT', 'PROPN', 'PUNCT', 'DET', 'NOUN', 'ADP', 'DET', 'NOUN', 'ADP', 'DET', 'NOUN', 'ADP', 'PROPN', 'PUNCT', 'ADP', 'DET', 'ADJ', 'NOUN', 'PUNCT']


In [8]:
label_names = list(set(tag for doc in train_labels for tag in doc))
label_names

['DET',
 'VERB',
 'ADP',
 'PROPN',
 'NOUN',
 'PRON',
 'CCONJ',
 'X',
 'PUNCT',
 'SCONJ',
 'ADV',
 'NUM',
 '_',
 'SYM',
 'PART',
 'ADJ',
 'INTJ',
 'AUX']

# STEP 3: Tokenization + Label Alignment

In [9]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [10]:
label2id = {label: i for i, label in enumerate(label_names)}
id2label = {i: label for label, i in label2id.items()}

In [12]:
def tokenize_and_align_labels(texts, labels):
    tokenized_inputs = tokenizer(
        texts,
        truncation=True,
        padding=True,
        is_split_into_words=True
    )

    aligned_labels = []

    for i, label in enumerate(labels):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label2id[label[word_idx]])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        aligned_labels.append(label_ids)

    tokenized_inputs["labels"] = aligned_labels
    return tokenized_inputs

In [13]:
train_encodings = tokenize_and_align_labels(train_texts, train_labels)
val_encodings = tokenize_and_align_labels(val_texts, val_labels)
test_encodings = tokenize_and_align_labels(test_texts, test_labels)

In [ ]:
print(train_encodings.keys())

# STEP 4: Model Setup (BERT for Token Classification)

In [14]:
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized be

In [16]:
data_collator = DataCollatorForTokenClassification(tokenizer)

In [15]:
import torch

class POSDataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __getitem__(self, idx):
        return {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}

    def __len__(self):
        return len(self.encodings["input_ids"])

train_dataset = POSDataset(train_encodings)
val_dataset = POSDataset(val_encodings)
test_dataset = POSDataset(test_encodings)

In [ ]:
print(train_dataset[0])

# STEP 5: Training (Hugging Face Trainer)

In [17]:
metric = evaluate.load("seqeval")

In [18]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [id2label[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    true_labels = [
        [id2label[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
    }

In [19]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=500,
    disable_tqdm=True
)

In [20]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [24]:
trainer.train()

{'loss': '0.02861', 'grad_norm': '1.025', 'learning_rate': '1.576e-05', 'epoch': '0.6378'}
{'eval_loss': '0.1207', 'eval_precision': '0.9679', 'eval_recall': '0.9693', 'eval_f1': '0.9686', 'eval_runtime': '12.12', 'eval_samples_per_second': '165.2', 'eval_steps_per_second': '10.4', 'epoch': '1'}
{'loss': '0.02901', 'grad_norm': '0.8838', 'learning_rate': '1.151e-05', 'epoch': '1.276'}
{'loss': '0.01823', 'grad_norm': '0.8103', 'learning_rate': '7.253e-06', 'epoch': '1.913'}
{'eval_loss': '0.1274', 'eval_precision': '0.9695', 'eval_recall': '0.9708', 'eval_f1': '0.9701', 'eval_runtime': '12.01', 'eval_samples_per_second': '166.6', 'eval_steps_per_second': '10.49', 'epoch': '2'}
{'loss': '0.01259', 'grad_norm': '1.027', 'learning_rate': '3.002e-06', 'epoch': '2.551'}
{'eval_loss': '0.1354', 'eval_precision': '0.9698', 'eval_recall': '0.9706', 'eval_f1': '0.9702', 'eval_runtime': '12.48', 'eval_samples_per_second': '160.4', 'eval_steps_per_second': '10.1', 'epoch': '3'}
{'train_runtime': 

TrainOutput(global_step=2352, training_loss=0.020639484025994127, metrics={'train_runtime': 1264.3859, 'train_samples_per_second': 29.763, 'train_steps_per_second': 1.86, 'train_loss': 0.020639484025994127, 'epoch': 3.0})

# STEP 6: Evaluation (Precision, Recall, F1)

In [25]:
results = trainer.evaluate()
results

{'eval_loss': '0.1354', 'eval_precision': '0.9698', 'eval_recall': '0.9706', 'eval_f1': '0.9702', 'eval_runtime': '11.95', 'eval_samples_per_second': '167.5', 'eval_steps_per_second': '10.55', 'epoch': '3'}


{'eval_loss': 0.13544973731040955,
 'eval_precision': 0.9698151175952711,
 'eval_recall': 0.9706289598455923,
 'eval_f1': 0.9702218680535167,
 'eval_runtime': 11.9455,
 'eval_samples_per_second': 167.51,
 'eval_steps_per_second': 10.548,
 'epoch': 3.0}

In [26]:
print("Precision:", results["eval_precision"])
print("Recall:", results["eval_recall"])
print("F1 Score:", results["eval_f1"])

Precision: 0.9698151175952711
Recall: 0.9706289598455923
F1 Score: 0.9702218680535167


# STEP 7: Inference (Custom Sentence Prediction)

In [27]:
def predict(sentence):
    tokens = sentence.split()

    inputs = tokenizer(
        tokens,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    inputs = {key: val.to(device) for key, val in inputs.items()}

    outputs = model(**inputs)
    predictions = outputs.logits.argmax(dim=2)

    predicted_labels = []

    # ✅ correct way
    word_ids = tokenizer(tokens, is_split_into_words=True).word_ids()

    previous_word_idx = None

    for idx, word_idx in enumerate(word_ids):
        if word_idx is None:
            continue
        elif word_idx != previous_word_idx:
            predicted_labels.append(id2label[predictions[0][idx].item()])
        previous_word_idx = word_idx

    return list(zip(tokens, predicted_labels))

In [28]:
sentence = "John works at Google in California"
output = predict(sentence)

for word, tag in output:
    print(f"{word} → {tag}")

John → PROPN
works → VERB
at → ADP
Google → PROPN
in → ADP
California → PROPN
